# Google - The Gemma 4 Developer Agent Competition: Complete EDA

This notebook explores the public data for the **Google - The Gemma 4 Developer Agent Competition**.

The goal is not just to describe files. The goal is to understand what the benchmark asks an agent to do, what kinds of software changes appear in the public task data, and what that implies for future submissions.


## 1. Executive Summary

This EDA answers four practical questions:

1. **What is in the public competition dataset?** We inspect task records, harness documentation, embeddings, graph files, snapshots, wheels, Docker/sandbox files, and sample submission files.
2. **What does the evaluation harness reward?** We extract the public evaluation contract: declarative submissions, tool-mediated repository repair, patch extraction, and verification.
3. **What kinds of bugs and patches appear in the public task data?** We analyze repositories, issue text, hints, source patches, test patches, touched paths, diff sizes, and transparent bug-category heuristics.
4. **What does this imply for agent design?** We convert observed patterns into prompt/config recommendations for future submissions.

The analysis is intentionally legitimate: it uses only public files mounted in the competition input and does not try to infer hidden tests beyond normal benchmark understanding.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import tarfile
import textwrap
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)

COMPETITION_SLUG = "gemma-4-developer-agent"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Environment and Data Discovery

Kaggle input folder names can vary depending on how the dataset is attached. The notebook therefore searches `/kaggle/input` recursively for competition-specific signals such as `tasks.jsonl`, `HARNESS_README.md`, graph files, embeddings, snapshots, wheels, and `sample_submission`.

For local development outside Kaggle, the code also supports a repository-local fallback path if present.

In [ ]:
def human_bytes(num: float) -> str:
    if pd.isna(num):
        return "n/a"
    units = ["B", "KB", "MB", "GB", "TB"]
    n = float(num)
    for unit in units:
        if abs(n) < 1024 or unit == units[-1]:
            return f"{n:,.1f} {unit}"
        n /= 1024


def safe_rel(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)


def infer_category(path: Path) -> str:
    p = str(path).lower()
    name = path.name.lower()
    suffixes = ''.join(path.suffixes).lower()
    if name == "tasks.jsonl":
        return "tasks"
    if name == "harness_readme.md":
        return "harness_docs"
    if "sample_submission" in p:
        return "sample_submission"
    if "/embeddings/" in p or suffixes.endswith(".npz"):
        return "embeddings"
    if "/graphs/" in p or suffixes.endswith(".json"):
        return "graphs_json"
    if suffixes.endswith(".tgz") or suffixes.endswith(".tar.gz"):
        return "repo_snapshots"
    if suffixes.endswith(".whl"):
        return "wheels"
    if "/docker/" in p or name.startswith("dockerfile"):
        return "docker"
    if "/sandbox/" in p:
        return "sandbox"
    if suffixes.endswith(('.yaml', '.yml')):
        return "yaml_config"
    if suffixes.endswith(('.md', '.txt')):
        return "text_docs"
    if suffixes.endswith(('.py',)):
        return "python_support"
    return "other"


def find_candidate_roots() -> list[Path]:
    roots = []
    kaggle_input = Path("/kaggle/input") 
    if kaggle_input.exists():
        roots.append(kaggle_input)

    # Local fallbacks for repo development. Avoid scanning the whole repository
    # because virtualenvs and experiment outputs can be large and irrelevant.
    local_candidates = [
        Path.cwd() / "competitions" / COMPETITION_SLUG / "data" / "extracted_minimal",
        Path.cwd() / "data" / "extracted_minimal",
    ]
    for root in local_candidates:
        if root.exists() and root not in roots:
            roots.append(root)
    return roots


def build_manifest(roots: list[Path]) -> pd.DataFrame:
    records = []
    seen = set()
    for root in roots:
        try:
            files = [p for p in root.rglob("*") if p.is_file()]
        except Exception:
            continue
        for path in files:
            key = path.resolve()
            if key in seen:
                continue
            seen.add(key)
            try:
                stat = path.stat()
            except OSError:
                continue
            records.append({
                "root": str(root),
                "path": str(path),
                "relative_path": safe_rel(path, root),
                "name": path.name,
                "suffix": ''.join(path.suffixes).lower() or path.suffix.lower(),
                "parent": path.parent.name,
                "category": infer_category(path),
                "size_bytes": stat.st_size,
                "size_mb": stat.st_size / (1024 ** 2),
            })
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["category", "relative_path"]).reset_index(drop=True)
    return df

roots = find_candidate_roots()
print("Candidate roots:")
for root in roots:
    print("-", root)

manifest = build_manifest(roots)
print(f"\nDiscovered {len(manifest):,} files totaling {human_bytes(manifest['size_bytes'].sum() if not manifest.empty else 0)}")
display(manifest.head(20))

In [ ]:
if manifest.empty:
    raise RuntimeError("No input files discovered. Attach the competition dataset in Kaggle or run from a repo with local data.")

category_summary = (
    manifest.groupby("category", as_index=False)
    .agg(files=("path", "count"), total_bytes=("size_bytes", "sum"), median_mb=("size_mb", "median"), max_mb=("size_mb", "max"))
    .sort_values("total_bytes", ascending=False)
)
category_summary["total_size"] = category_summary["total_bytes"].map(human_bytes)
display(category_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=category_summary, y="category", x="files", ax=axes[0], color="#4C78A8")
axes[0].set_title("File count by category")
axes[0].set_xlabel("Files")
axes[0].set_ylabel("")

sns.barplot(data=category_summary, y="category", x="total_bytes", ax=axes[1], color="#F58518")
axes[1].set_title("Total bytes by category")
axes[1].set_xlabel("Bytes")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

largest = manifest.sort_values("size_bytes", ascending=False).head(25).copy()
largest["size"] = largest["size_bytes"].map(human_bytes)
display(largest[["relative_path", "category", "size"]])

## 3. Competition Harness and Evaluation Contract

The harness documentation matters because it tells us what an agent can actually do. In this competition, the submission is a declarative agent package, the benchmark runs the agent inside a sandboxed repository, extracts a patch, and verifies that patch in a fresh environment.

The following cells locate `HARNESS_README.md`, extract relevant snippets, and convert them into practical constraints.

In [ ]:
def find_first_manifest(category: str | None = None, name: str | None = None, contains: str | None = None) -> Path | None:
    df = manifest
    if category is not None:
        df = df[df["category"] == category]
    if name is not None:
        df = df[df["name"].str.lower() == name.lower()]
    if contains is not None:
        df = df[df["relative_path"].str.lower().str.contains(contains.lower(), regex=False)]
    if df.empty:
        return None
    return Path(df.sort_values("size_bytes", ascending=False).iloc[0]["path"])

harness_path = find_first_manifest(name="HARNESS_README.md")
print("HARNESS_README:", harness_path)

harness_text = ""
if harness_path and harness_path.exists():
    harness_text = harness_path.read_text(errors="replace")
    print(f"Characters: {len(harness_text):,}")
    print(harness_text[:2500])
else:
    print("HARNESS_README.md not found.")

In [ ]:
KEYWORDS = [
    "Submission Directory Layout", "Built-In Tools", "Evaluation Lifecycle", "Budgets", "Operational Limits",
    "Patch Extraction", "verify_task", "submit_patch", "max_tool_calls", "max_turns", "Never", "tools",
]

def keyword_context(text: str, keyword: str, window: int = 700) -> str | None:
    idx = text.lower().find(keyword.lower())
    if idx < 0:
        return None
    start = max(0, idx - window // 3)
    end = min(len(text), idx + window)
    snippet = text[start:end]
    return re.sub(r"\n{3,}", "\n\n", snippet).strip()

if harness_text:
    for kw in KEYWORDS:
        snippet = keyword_context(harness_text, kw)
        if snippet:
            display(Markdown(f"### Keyword hit: `{kw}`\n\n```text\n{snippet[:1800]}\n```"))

    practical_constraints = [
        "Submissions are declarative: agent behavior is mostly controlled through YAML, prompts, tools, generation config, and adapters.",
        "The agent must produce a source patch; empty patches or over-analysis are unlikely to score.",
        "The benchmark separates agent execution from verification, so local mutations to tests should not be part of the final useful patch.",
        "Tool budgets and context limits make fast targeting and targeted verification more valuable than full-suite exploration.",
        "The public task data can reveal repair patterns, but hidden evaluation should be treated as unseen tasks from the same general repair setting.",
    ]
    display(Markdown("### Practical constraints for agent design\n" + "\n".join(f"- {x}" for x in practical_constraints)))

## 4. Task Dataset Schema

`tasks.jsonl` is the key tabular dataset. Each row describes a public repair task: repository, base commit, problem statement, hints, source patch, and test patch.

We load it defensively and inspect schema, missing values, duplicates, repositories, and representative examples.

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    bad = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                bad.append((i, str(e), line[:200]))
    if bad:
        print(f"Warning: {len(bad)} JSONL lines failed to parse")
        display(pd.DataFrame(bad, columns=["line", "error", "preview"]).head())
    return pd.DataFrame(rows)

tasks_path = find_first_manifest(name="tasks.jsonl")
print("tasks.jsonl:", tasks_path)
if tasks_path is None:
    raise RuntimeError("tasks.jsonl not found in discovered files.")

tasks = load_jsonl(tasks_path)
print("Shape:", tasks.shape)
display(tasks.head(3))

schema = pd.DataFrame({
    "column": tasks.columns,
    "dtype": [str(tasks[c].dtype) for c in tasks.columns],
    "non_null": [tasks[c].notna().sum() for c in tasks.columns],
    "missing": [tasks[c].isna().sum() for c in tasks.columns],
    "unique": [tasks[c].nunique(dropna=True) for c in tasks.columns],
})
display(schema)

In [ ]:
print("Duplicate instance_id rows:", tasks.duplicated("instance_id").sum() if "instance_id" in tasks else "n/a")
print("Fully duplicated rows:", tasks.duplicated().sum())

if "created_at" in tasks.columns:
    tasks["created_at_dt"] = pd.to_datetime(tasks["created_at"], errors="coerce", utc=True)
    print("Date range:", tasks["created_at_dt"].min(), "to", tasks["created_at_dt"].max())

repo_counts = tasks["repo"].value_counts().rename_axis("repo").reset_index(name="tasks") if "repo" in tasks else pd.DataFrame()
display(repo_counts)

if not repo_counts.empty:
    ax = sns.barplot(data=repo_counts, y="repo", x="tasks", color="#4C78A8")
    ax.set_title("Tasks by repository")
    ax.set_xlabel("Task count")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 5. Repository and Instance Distribution

The repository distribution tells us whether the public data is broad or concentrated. Concentration matters because prompt heuristics may be rewarded when they match common frameworks, but a robust agent still needs general repair behavior for hidden tasks.

In [ ]:
if "instance_id" in tasks.columns:
    tasks["instance_prefix"] = tasks["instance_id"].astype(str).str.extract(r"^([^_]+)", expand=False).fillna("unknown")
    prefix_counts = tasks["instance_prefix"].value_counts().rename_axis("prefix").reset_index(name="tasks")
    display(prefix_counts)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.countplot(data=tasks, y="instance_prefix", order=tasks["instance_prefix"].value_counts().index, ax=axes[0], color="#54A24B")
    axes[0].set_title("Instance prefix distribution")
    axes[0].set_xlabel("Tasks")
    axes[0].set_ylabel("")

    if "repo" in tasks.columns:
        ctab = pd.crosstab(tasks["repo"], tasks["instance_prefix"])
        sns.heatmap(ctab, annot=True, fmt="d", cmap="Blues", ax=axes[1])
        axes[1].set_title("Repo vs instance prefix")
        axes[1].set_xlabel("Prefix")
        axes[1].set_ylabel("Repo")
    plt.tight_layout()
    plt.show()

if "base_commit" in tasks.columns:
    print("Unique base commits:", tasks["base_commit"].nunique())
    display(tasks.groupby("repo")["base_commit"].nunique().sort_values(ascending=False).rename("unique_base_commits").reset_index() if "repo" in tasks else tasks["base_commit"].value_counts().head())

## 6. Text Length and Prompt Burden

Developer-agent tasks are constrained by tool calls and context window. Very long problem statements, hints, patches, or test patches can consume attention and context budget. This section computes character lengths and rough token estimates to understand how much information the agent needs to triage.

In [ ]:
TEXT_COLUMNS = [c for c in ["problem_statement", "hints_text", "patch", "test_patch"] if c in tasks.columns]

def approx_tokens(text: object) -> int:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return 0
    s = str(text)
    return max(1, math.ceil(len(s) / 4)) if s else 0

for col in TEXT_COLUMNS:
    tasks[f"{col}_chars"] = tasks[col].fillna("").astype(str).str.len()
    tasks[f"{col}_approx_tokens"] = tasks[col].map(approx_tokens)

length_cols = [f"{c}_chars" for c in TEXT_COLUMNS] + [f"{c}_approx_tokens" for c in TEXT_COLUMNS]
summary = tasks[length_cols].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T
summary["mean"] = summary["mean"].round(1)
display(summary)

fig, axes = plt.subplots(len(TEXT_COLUMNS), 2, figsize=(15, max(4, 3.5 * len(TEXT_COLUMNS))))
if len(TEXT_COLUMNS) == 1:
    axes = np.array([axes])
for i, col in enumerate(TEXT_COLUMNS):
    sns.histplot(tasks[f"{col}_chars"], bins=30, ax=axes[i, 0], color="#4C78A8")
    axes[i, 0].set_title(f"{col}: character length")
    axes[i, 0].set_xlabel("Characters")
    sns.boxplot(x=tasks[f"{col}_chars"], ax=axes[i, 1], color="#F58518")
    axes[i, 1].set_title(f"{col}: outliers")
    axes[i, 1].set_xlabel("Characters")
plt.tight_layout()
plt.show()

for col in ["problem_statement", "patch", "test_patch"]:
    if col in tasks.columns:
        display(Markdown(f"### Longest `{col}` examples"))
        show_cols = ["instance_id", "repo", f"{col}_chars", col]
        display(tasks.sort_values(f"{col}_chars", ascending=False)[show_cols].head(3))

## 7. Patch and Test Patch Analysis

The public patches reveal the shape of expected fixes. We parse unified diffs to estimate changed files, added/deleted lines, file extensions, and top paths. This does not tell us hidden answers, but it gives a realistic prior for agent behavior.

In [ ]:
DIFF_FILE_RE = re.compile(r"^\+\+\+ b/(.+)$")
DIFF_OLD_RE = re.compile(r"^--- a/(.+)$")


def parse_unified_diff(diff_text: object) -> dict:
    s = "" if diff_text is None or (isinstance(diff_text, float) and pd.isna(diff_text)) else str(diff_text)
    files = []
    additions = 0
    deletions = 0
    hunks = 0
    current = None
    for line in s.splitlines():
        if line.startswith("@@"):
            hunks += 1
        m = DIFF_FILE_RE.match(line)
        if m:
            current = m.group(1)
            files.append(current)
            continue
        if line.startswith("+") and not line.startswith("+++"):
            additions += 1
        elif line.startswith("-") and not line.startswith("---"):
            deletions += 1
    exts = [Path(f).suffix.lower() or "[no_ext]" for f in files]
    return {
        "changed_files": len(set(files)),
        "file_list": sorted(set(files)),
        "extensions": exts,
        "additions": additions,
        "deletions": deletions,
        "hunks": hunks,
    }

for diff_col in ["patch", "test_patch"]:
    if diff_col in tasks.columns:
        parsed = tasks[diff_col].map(parse_unified_diff)
        for key in ["changed_files", "additions", "deletions", "hunks"]:
            tasks[f"{diff_col}_{key}"] = parsed.map(lambda x: x[key])
        tasks[f"{diff_col}_file_list"] = parsed.map(lambda x: x["file_list"])
        tasks[f"{diff_col}_extensions"] = parsed.map(lambda x: x["extensions"])

metric_cols = [c for c in tasks.columns if re.search(r"_(changed_files|additions|deletions|hunks)$", c)]
display(tasks[["instance_id", "repo"] + metric_cols].head())
display(tasks[metric_cols].describe(percentiles=[.5, .75, .9, .95, .99]).T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
plot_pairs = [
    ("patch_changed_files", "Source patch changed files"),
    ("patch_additions", "Source patch additions"),
    ("test_patch_changed_files", "Test patch changed files"),
    ("test_patch_additions", "Test patch additions"),
]
for ax, (col, title) in zip(axes.ravel(), plot_pairs):
    if col in tasks.columns:
        sns.histplot(tasks[col], bins=30, ax=ax, color="#4C78A8")
        ax.set_title(title)
        ax.set_xlabel(col)
plt.tight_layout()
plt.show()

for diff_col in ["patch", "test_patch"]:
    file_col = f"{diff_col}_file_list"
    ext_col = f"{diff_col}_extensions"
    if file_col in tasks.columns:
        file_counts = Counter(f for files in tasks[file_col] for f in files)
        ext_counts = Counter(e for exts in tasks[ext_col] for e in exts)
        display(Markdown(f"### Top changed files for `{diff_col}`"))
        display(pd.DataFrame(file_counts.most_common(20), columns=["file", "count"]))
        display(Markdown(f"### Top extensions for `{diff_col}`"))
        display(pd.DataFrame(ext_counts.most_common(20), columns=["extension", "count"]))

## 8. Bug/Task Category Heuristics

A useful EDA can classify tasks with transparent, imperfect heuristics. These categories are not labels from Kaggle; they are interpretable signals derived from issue text, hints, patch paths, and diff content.

The goal is to identify prompt heuristics that deserve space in future submissions.

In [ ]:
CATEGORY_PATTERNS = {
    "validation_or_parsing": [r"validat", r"parse", r"parser", r"invalid", r"pydantic", r"schema", r"typeerror", r"valueerror"],
    "serialization_schema": [r"json", r"yaml", r"serialize", r"deserialize", r"schema", r"field", r"model_dump", r"encoder"],
    "cli_config": [r"cli", r"command", r"argument", r"option", r"flag", r"config", r"env var", r"settings"],
    "imports_paths_packaging": [r"import", r"module", r"path", r"package", r"relative", r"namespace", r"distribution"],
    "compatibility_versioning": [r"version", r"compat", r"deprecated", r"python 3", r"starlette", r"pydantic v", r"requests", r"httpx"],
    "error_messages_status": [r"error message", r"exception", r"traceback", r"status code", r"warning", r"raise", r"assert"],
    "async_network": [r"async", r"await", r"http", r"request", r"response", r"client", r"stream", r"timeout"],
    "rendering_output_format": [r"render", r"format", r"markup", r"rich", r"console", r"output", r"display", r"ansi"],
    "docs_release_tooling": [r"release", r"changelog", r"docs", r"documentation", r"script", r"prepare_release"],
    "tests_behavior": [r"test", r"pytest", r"fixture", r"regression"],
}


def combined_task_text(row: pd.Series) -> str:
    parts = []
    for col in ["repo", "instance_id", "problem_statement", "hints_text", "patch", "test_patch"]:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return "\n".join(parts).lower()


def classify_task(row: pd.Series) -> list[str]:
    text = combined_task_text(row)
    labels = []
    for label, patterns in CATEGORY_PATTERNS.items():
        if any(re.search(p, text, flags=re.I) for p in patterns):
            labels.append(label)
    return labels or ["other"]

tasks["heuristic_categories"] = tasks.apply(classify_task, axis=1)
category_counts = Counter(cat for cats in tasks["heuristic_categories"] for cat in cats)
cat_df = pd.DataFrame(category_counts.most_common(), columns=["category", "tasks"])
display(cat_df)

plt.figure(figsize=(11, 6))
sns.barplot(data=cat_df, y="category", x="tasks", color="#4C78A8")
plt.title("Heuristic bug/task category distribution")
plt.xlabel("Tasks matched")
plt.ylabel("")
plt.tight_layout()
plt.show()

if "repo" in tasks.columns:
    exploded = tasks[["repo", "heuristic_categories"]].explode("heuristic_categories")
    ctab = pd.crosstab(exploded["repo"], exploded["heuristic_categories"])
    plt.figure(figsize=(14, max(4, .45 * len(ctab))))
    sns.heatmap(ctab, annot=True, fmt="d", cmap="Blues")
    plt.title("Heuristic categories by repository")
    plt.xlabel("Category")
    plt.ylabel("Repository")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
def compact_text(s: object, max_chars: int = 900) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= max_chars else s[:max_chars] + " ..."

for category in cat_df["category"].head(8):
    subset = tasks[tasks["heuristic_categories"].map(lambda xs: category in xs)]
    if subset.empty:
        continue
    display(Markdown(f"### Representative examples: `{category}`"))
    cols = [c for c in ["instance_id", "repo", "problem_statement", "hints_text", "patch_changed_files", "patch_file_list"] if c in subset.columns]
    examples = subset.sample(min(3, len(subset)), random_state=RANDOM_SEED)[cols].copy()
    for c in ["problem_statement", "hints_text"]:
        if c in examples.columns:
            examples[c] = examples[c].map(lambda x: compact_text(x, 600))
    display(examples)

## 9. Embeddings, Graphs, Snapshots, Wheels, and Docker Files

The competition data is not only `tasks.jsonl`. It includes support artifacts used by the harness and code-intelligence tools: embeddings, repository graphs, snapshots, wheels, Docker files, sandbox setup, and sample submission files.

We inspect these safely without extracting large archives by default.

In [ ]:
# Embeddings: inspect metadata for a sample of .npz files.
npz_files = manifest[manifest["suffix"].str.endswith(".npz", na=False)].sort_values("size_bytes", ascending=False)
print(f"Embedding .npz files discovered: {len(npz_files):,}")

def inspect_npz(path: Path) -> list[dict]:
    records = []
    try:
        with np.load(path, allow_pickle=False) as data:
            for key in data.files:
                arr = data[key]
                records.append({
                    "file": path.name,
                    "array": key,
                    "shape": tuple(arr.shape),
                    "dtype": str(arr.dtype),
                    "size_mb": arr.nbytes / (1024 ** 2),
                })
    except Exception as e:
        records.append({"file": path.name, "array": "[error]", "shape": str(e), "dtype": "", "size_mb": np.nan})
    return records

npz_records = []
for p in npz_files["path"].head(12):
    npz_records.extend(inspect_npz(Path(p)))
npz_meta = pd.DataFrame(npz_records)
display(npz_meta)

In [ ]:
# Graph JSON files: inspect top-level schema and approximate node/edge-like structures.
graph_files = manifest[(manifest["category"] == "graphs_json") | (manifest["relative_path"].str.lower().str.contains("/graphs/", regex=False))]
print(f"Graph JSON files discovered: {len(graph_files):,}")

def summarize_json(path: Path) -> dict:
    record = {"file": path.name, "size_mb": path.stat().st_size / (1024 ** 2)}
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            obj = json.load(f)
        record["top_type"] = type(obj).__name__
        if isinstance(obj, dict):
            record["top_keys"] = list(obj.keys())[:20]
            for key, value in obj.items():
                kl = str(key).lower()
                if isinstance(value, list) and any(word in kl for word in ["node", "vertex", "edge", "symbol", "chunk"]):
                    record[f"len_{key}"] = len(value)
                elif isinstance(value, dict) and any(word in kl for word in ["node", "vertex", "edge", "symbol", "chunk"]):
                    record[f"len_{key}"] = len(value)
        elif isinstance(obj, list):
            record["items"] = len(obj)
            record["sample_type"] = type(obj[0]).__name__ if obj else None
    except Exception as e:
        record["error"] = repr(e)
    return record

graph_meta = pd.DataFrame([summarize_json(Path(p)) for p in graph_files["path"].head(10)])
display(graph_meta)

In [ ]:
# Repository snapshots: inspect tar metadata and sample members without extracting whole archives.
tar_files = manifest[manifest["suffix"].str.endswith((".tgz", ".tar.gz"), na=False)].sort_values("size_bytes", ascending=False)
print(f"Snapshot archives discovered: {len(tar_files):,}")

def inspect_tar(path: Path, sample_n: int = 40) -> dict:
    rec = {"file": path.name, "size_mb": path.stat().st_size / (1024 ** 2)}
    try:
        with tarfile.open(path, "r:*") as tf:
            members = tf.getmembers()
            rec["members"] = len(members)
            rec["dirs"] = sum(1 for m in members if m.isdir())
            rec["files"] = sum(1 for m in members if m.isfile())
            rec["uncompressed_mb_est"] = sum(getattr(m, "size", 0) for m in members if m.isfile()) / (1024 ** 2)
            rec["sample_paths"] = [m.name for m in members[:sample_n]]
    except Exception as e:
        rec["error"] = repr(e)
    return rec

tar_meta = pd.DataFrame([inspect_tar(Path(p)) for p in tar_files["path"].head(5)])
display(tar_meta.drop(columns=["sample_paths"], errors="ignore"))
for _, row in tar_meta.iterrows():
    display(Markdown(f"### Sample members: `{row.get('file')}`"))
    for item in row.get("sample_paths", [])[:40]:
        print(item)

In [ ]:
# Wheels: parse package names and versions from wheel filenames.
wheel_files = manifest[manifest["suffix"].str.endswith(".whl", na=False)].copy()
print(f"Wheel files discovered: {len(wheel_files):,}")

WHEEL_RE = re.compile(r"^(?P<name>.+?)-(?P<version>[^-]+)-(?P<build_or_py>[^-]+)(?:-(?P<abi>[^-]+)-(?P<platform>.+))?\.whl$")

def parse_wheel_name(name: str) -> dict:
    m = WHEEL_RE.match(name)
    if not m:
        return {"package": name.removesuffix(".whl"), "version": None, "python_tag": None, "abi": None, "platform": None}
    d = m.groupdict()
    return {
        "package": d["name"].replace("_", "-"),
        "version": d["version"],
        "python_tag": d["build_or_py"],
        "abi": d.get("abi"),
        "platform": d.get("platform"),
    }

if not wheel_files.empty:
    parsed_wheels = pd.concat([wheel_files.reset_index(drop=True), pd.DataFrame([parse_wheel_name(n) for n in wheel_files["name"]])], axis=1)
    display(parsed_wheels[["relative_path", "package", "version", "python_tag", "abi", "platform", "size_mb"]].head(30))
    pkg_summary = parsed_wheels.groupby("package", as_index=False).agg(versions=("version", "nunique"), files=("name", "count"), total_mb=("size_mb", "sum")).sort_values("files", ascending=False)
    display(pkg_summary.head(30))

    top_size = parsed_wheels.sort_values("size_mb", ascending=False).head(20)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top_size, y="package", x="size_mb", color="#F58518")
    plt.title("Largest wheel packages")
    plt.xlabel("Size (MB)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
# Docker/sandbox/sample submission files: show trees and read small key files.
def print_tree(paths: list[Path], max_lines: int = 120) -> None:
    for i, p in enumerate(paths[:max_lines]):
        print(p)
    if len(paths) > max_lines:
        print(f"... {len(paths) - max_lines} more")

def read_small(path: Path, max_chars: int = 6000) -> str:
    try:
        text = path.read_text(errors="replace")
    except Exception as e:
        return f"[Could not read: {e}]"
    return text[:max_chars] + ("\n...[truncated]" if len(text) > max_chars else "")

for cat in ["docker", "sandbox", "sample_submission", "yaml_config", "python_support"]:
    subset = manifest[manifest["category"] == cat].sort_values("relative_path")
    if subset.empty:
        continue
    display(Markdown(f"### `{cat}` files"))
    display(subset[["relative_path", "size_bytes", "category"]].head(40))

sample_subset = manifest[manifest["relative_path"].str.lower().str.contains("sample_submission", regex=False)].sort_values("relative_path")
if not sample_subset.empty:
    display(Markdown("## Sample submission file previews"))
    for p in sample_subset["path"].head(10):
        path = Path(p)
        display(Markdown(f"### `{safe_rel(path, Path(sample_subset.iloc[0]['root']))}`"))
        print(read_small(path, 4000))